# DSC540-T302 Data Preparation Term Project
## Milestone 4 : Cleaning/Formatting API Data

In [78]:
# INTIAL STEP TO MAKE THE API CALL AND STORE THE DATA IN A DATAFRAME

#Import necessary libraries
import requests
import pandas as pd
import json

# Load the countries dataframe (created using a previous milestone dataframe)
df = pd.read_csv('country_data.csv') 

# Extract the country names column
countries = df['Country Name'].unique()  # Use .unique() to avoid duplicate API calls since original DF repeats countries per year

# Load API key from JSON file
with open('APIKeys.json') as x:
    keys = json.load(x)
    apininja = keys['Ninjaapi']

# Function to fetch country data
def get_country_data(country):
    api_url = f'https://api.api-ninjas.com/v1/country?name={country}'
    response = requests.get(api_url, headers={'X-Api-Key': apininja})

    if response.status_code == requests.codes.ok:
        data = response.json()
        if data:  # Only add if data exists
            country_data = data[0]  # Extract first item from list
            country_data["name"] = country  # Ensure country name is explicitly included
            
            # Flatten currency dictionary since it's nested
            currency = country_data.pop("currency", {})  # Remove currency from original dict
            country_data["currency_code"] = currency.get("code", None)
            country_data["currency_name"] = currency.get("name", None)
            return country_data  # Return full country data
        else:
            print(f"Skipping {country}: No data found.")
            return None  # Return None to exclude this country
    else:
        print(f"Error fetching {country}: {response.status_code}")
        return None  # Return None to exclude this country


# Fetch data for all countries and store in a list
data = [get_country_data(country) for country in countries]



Skipping South Korea: No data found.
Skipping UAE: No data found.
Skipping Democratic Republic Of Congo: No data found.
Skipping West Bank And Gaza: No data found.
Skipping Lao PDR: No data found.
Skipping Republic Of Congo: No data found.
Skipping St. Lucia: No data found.
Skipping St. Kitts And Nevis: No data found.
Skipping St. Vincent And The Grenadines: No data found.
Skipping St. Martin (French Part): No data found.
Skipping Channel Islands: No data found.
Skipping Virgin Islands (U.S.): No data found.


In [80]:
# Remove countries with no data
data = [entry for entry in data if entry is not None]  

# Convert list of dicts to a DataFrame
country_data_df = pd.DataFrame(data)

# Display dataframe
country_data_df

,gdp,sex_ratio,surface_area,life_expectancy_male,unemployment,imports,homicide_rate,iso2,employment_services,employment_industry,...,pop_growth,region,pop_density,internet_users,gdp_per_capita,fertility,refugees,primary_school_enrollment_male,currency_code,currency_name
0,20580223.0,97.9,9833517.0,76.3,3.9,2567490.0,5.0,US,79.0,19.7,...,0.6,Northern America,36.2,87.3,62917.9,1.8,1043.2,102.2,USD,Us Dollar
1,13608152.0,105.3,9600000.0,74.5,4.4,2070150.0,0.5,CN,47.1,28.2,...,0.5,Eastern Asia,153.3,54.3,9531.9,1.7,322.4,99.7,CNY,Yuan Renminbi
2,3949549.0,97.8,357376.0,78.7,3.0,1240700.0,0.9,DE,72.1,26.8,...,0.5,Western Europe,240.4,89.7,47513.7,1.6,1461.0,103.9,EUR,Euro
3,4971323.0,95.4,377930.0,81.3,2.3,720895.0,0.3,JP,72.6,24.1,...,-0.2,Eastern Asia,346.9,91.3,39082.1,1.4,31.5,NaN,JPY,Yen
4,2779352.0,108.2,3287263.0,68.1,5.4,478884.0,3.1,IN,32.3,26.2,...,1.0,Southern Asia,464.1,34.4,2054.8,2.2,206.6,105.6,INR,Indian Rupee
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,16628.0,100.2,185180.0,65.9,8.4,980.0,0.9,SY,63.0,26.5,...,-0.6,Western Asia,95.3,34.3,981.3,2.8,6386.7,83.1,SYP,Syrian Pound
193,NaN,107.1,457.0,74.9,11.2,5734.0,NaN,MP,NaN,NaN,...,0.2,Micronesia,120.3,NaN,NaN,1.6,NaN,NaN,USD,Us Dollar
194,6796.0,98.4,160.0,NaN,2.6,NaN,2.6,LI,NaN,NaN,...,0.7,Western Europe,241.5,98.1,179258.2,1.5,0.2,106.3,CHF,Swiss Franc
195,NaN,101.8,549.0,76.5,5.7,NaN,2.5,GU,85.5,14.3,...,0.8,Micronesia,312.5,80.5,NaN,2.3,NaN,NaN,USD,Us Dollar


In [82]:
# Check that the country names are in the DF
print(country_data_df.name)

0                 United States
1                         China
2                       Germany
3                         Japan
4                         India
                 ...           
192        Syrian Arab Republic
193    Northern Mariana Islands
194               Liechtenstein
195                        Guam
196                      Bhutan
Name: name, Length: 197, dtype: object


In [84]:
#Perform cleaning and transformations

# STEP 1 : Handle missing numeric values
# Fill with median as a way to prevent skewing the data during analysis.
country_data_df.fillna(country_data_df.median(numeric_only=True), inplace=True)

In [86]:
# STEP 2: Convert urban_population_growth, pop_growth, gdp_growth, fertility, and infant_mortality into perecentages
#Currently these columns are stored as a decimal, so we will multiply it by 100 for better readability. (Repeat step for 6 different columns)
country_data_df["gdp_growth"] = country_data_df["gdp_growth"] * 100
country_data_df["pop_growth"] = country_data_df["pop_growth"] * 100
country_data_df["urban_population_growth"] = country_data_df["urban_population_growth"] * 100
country_data_df["infant_mortality"] = country_data_df["infant_mortality"] * 100
country_data_df["fertility"] = country_data_df["fertility"] * 100
country_data_df["homicide_rate"] = country_data_df["homicide_rate"] * 100



In [88]:
#Check one of them:
country_data_df["pop_growth"]

0       60.0
1       50.0
2       50.0
3      -20.0
4      100.0
       ...  
192    -60.0
193     20.0
194     70.0
195     80.0
196    120.0
Name: pop_growth, Length: 197, dtype: float64

In [90]:
# STEP3 : Check and update data types
#Check the data type of entire DataFrame:
print(country_data_df.dtypes)


gdp                                   float64
sex_ratio                             float64
surface_area                          float64
life_expectancy_male                  float64
unemployment                          float64
imports                               float64
homicide_rate                         float64
iso2                                   object
employment_services                   float64
employment_industry                   float64
urban_population_growth               float64
secondary_school_enrollment_female    float64
employment_agriculture                float64
capital                                object
co2_emissions                         float64
forested_area                         float64
tourists                              float64
exports                               float64
life_expectancy_female                float64
post_secondary_enrollment_female      float64
post_secondary_enrollment_male        float64
primary_school_enrollment_female  

In [92]:
# All numberical columns appear as float already so we can move on to the next step

#STEP 4: Narrow down columns to keep

# List of columns to keep
columns_to_keep = ["name", "region", "population", "urban_population", "pop_density", 
                   "life_expectancy_male", "life_expectancy_female", "internet_users", "tourists"]

# Create a new dataframe with only these columns
filtered_df = country_data_df[columns_to_keep]
filtered_df

,name,region,population,urban_population,pop_density,life_expectancy_male,life_expectancy_female,internet_users,tourists
0,United States,Northern America,331003.0,82.5,36.2,76.3,81.3,87.3,79746.0
1,China,Eastern Asia,1439324.0,60.3,153.3,74.5,79.0,54.3,62900.0
2,Germany,Western Europe,83784.0,77.4,240.4,78.7,83.6,89.7,38881.0
3,Japan,Eastern Asia,126476.0,91.7,346.9,81.3,87.5,91.3,31192.0
4,India,Southern Asia,1380004.0,34.5,464.1,68.1,70.5,34.4,17423.0
...,...,...,...,...,...,...,...,...,...
192,Syrian Arab Republic,Western Asia,17501.0,54.8,95.3,65.9,77.7,34.3,5070.0
193,Northern Mariana Islands,Micronesia,55.0,91.7,120.3,74.9,77.9,63.2,517.0
194,Liechtenstein,Western Europe,39.0,14.4,241.5,71.1,77.5,98.1,85.0
195,Guam,Micronesia,169.0,94.9,312.5,76.5,83.3,80.5,1549.0


In [94]:
# STEP 5: Rename columns for added clarity 
#name to country_name
#urban_population to urban_population_percentage
#internet_users to #internet_users_percentage

filtered_df = filtered_df.copy() #Create copy (helped with removing wanring)

filtered_df.rename(columns={
    "name": "country_name",
    "urban_population": "urban_population_percentage",
    "pop_density": "population_density",
    "internet_users": "internet_users_percentage"
}, inplace=True)

filtered_df

,country_name,region,population,urban_population_percentage,population_density,life_expectancy_male,life_expectancy_female,internet_users_percentage,tourists
0,United States,Northern America,331003.0,82.5,36.2,76.3,81.3,87.3,79746.0
1,China,Eastern Asia,1439324.0,60.3,153.3,74.5,79.0,54.3,62900.0
2,Germany,Western Europe,83784.0,77.4,240.4,78.7,83.6,89.7,38881.0
3,Japan,Eastern Asia,126476.0,91.7,346.9,81.3,87.5,91.3,31192.0
4,India,Southern Asia,1380004.0,34.5,464.1,68.1,70.5,34.4,17423.0
...,...,...,...,...,...,...,...,...,...
192,Syrian Arab Republic,Western Asia,17501.0,54.8,95.3,65.9,77.7,34.3,5070.0
193,Northern Mariana Islands,Micronesia,55.0,91.7,120.3,74.9,77.9,63.2,517.0
194,Liechtenstein,Western Europe,39.0,14.4,241.5,71.1,77.5,98.1,85.0
195,Guam,Micronesia,169.0,94.9,312.5,76.5,83.3,80.5,1549.0


In [96]:
#STEP 6: Add a new calculated column for "Tourism Rate" by taking total tourists divided by total population

filtered_df["tourism_rate"] = filtered_df["tourists"] / filtered_df["population"]

# Replace infinite values (in case population is 0) with NaN
filtered_df.replace([float('inf'), -float('inf')], pd.NA, inplace=True)

filtered_df


,country_name,region,population,urban_population_percentage,population_density,life_expectancy_male,life_expectancy_female,internet_users_percentage,tourists,tourism_rate
0,United States,Northern America,331003.0,82.5,36.2,76.3,81.3,87.3,79746.0,0.240922
1,China,Eastern Asia,1439324.0,60.3,153.3,74.5,79.0,54.3,62900.0,0.043701
2,Germany,Western Europe,83784.0,77.4,240.4,78.7,83.6,89.7,38881.0,0.464062
3,Japan,Eastern Asia,126476.0,91.7,346.9,81.3,87.5,91.3,31192.0,0.246624
4,India,Southern Asia,1380004.0,34.5,464.1,68.1,70.5,34.4,17423.0,0.012625
...,...,...,...,...,...,...,...,...,...,...
192,Syrian Arab Republic,Western Asia,17501.0,54.8,95.3,65.9,77.7,34.3,5070.0,0.289698
193,Northern Mariana Islands,Micronesia,55.0,91.7,120.3,74.9,77.9,63.2,517.0,9.400000
194,Liechtenstein,Western Europe,39.0,14.4,241.5,71.1,77.5,98.1,85.0,2.179487
195,Guam,Micronesia,169.0,94.9,312.5,76.5,83.3,80.5,1549.0,9.165680


## Ethical Implications  

During this data transformation process, several steps were applied, including removing countries with no data, handling missing values by filling numeric fields, converting decimals to percentages for readability, and adding a calculated tourism rate. These changes were made to improve data usability in this project while ensuring consistency. However, these transformations come with some ethical considerations. For example, imputing missing values with the median assumes some amount of uniformity across countries, which may not truly reflect economic or demographic realities. Additionally, removing countries with no data could introduce bias by excluding regions that lack reporting capabilities. Given that this data originates from API Ninjas, it is important to see whether the source adheres to legal and regulatory guidelines related to demographic and economic data, such as those set by the United Nations, World Bank, or national statistics offices. The API Ninjas FAQ page mentions that their data engineers spend a lot of time curating and vetting the data sources, but they do not list out all of their sources so this is something to consider if using the data for any formal research. The calculated tourism rate may also be misleading if tourist data was collected using different methodologies across countries, affecting any analysis that will be performed using this metric. Furthermore, assumptions about urban population percentages and internet users could reinforce disparities if the data lacks representation for marginalized regions. To mitigate these ethical concerns, users should acknowledge potential biases when interpreting results, supplement the dataset with additional reputable sources if necessary, and ensure transparency in the transformation steps and further analysis to avoid misleading conclusions.

In [98]:
# ADDING CODE TO LOAD MILESTONE 4 (API) DF INTO SQLite

import sqlite3
import pandas as pd

# Connect to (or create) the SQLite database
conn = sqlite3.connect("combined_data.db")  # Ensure all notebooks use the same filename

# Write the DataFrame to a table in SQLite
filtered_df.to_sql("country_api", conn, if_exists="replace", index=False)

# Close the connection
conn.close()